# Static models for NOAA and OFAM3 data

Datasets:

1. NOAA: Weekly mean sea-surface temperature at 1° resolution, as given by the NOAA Optimum Interpolation SST V2 dataset [[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html) 

    - specifically, using the version taken from the Williams et al. pyshred implementation [[2]](https://github.com/Jan-Williams/pyshred/blob/main/Data/SST_data.mat)

2. OFAM3: ME4OH full-field dataset at 1/4° resolution generated from the OFAM3 numerical model [[3]](https://data.csiro.au/collection/csiro%3A60826v9). 

Models: 
- Static SHRED & SDN from [[4](https://github.com/Jan-Williams/pyshred/blob/main/models.py), [5](https://royalsocietypublishing.org/rspa/article/480/2298/20240054/66770/Sensing-with-shallow-recurrent-decoder)]

In [ ]:
# Standard imports
import json
import os
import random

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

# Local imports
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py
from helper_functions import noaa_data, ofam3_data
from helper_functions.plotting_functions import plot_sensors, plot_sst_map
from helper_functions.pod import qr_place
from helper_functions.preprocess_SDN import create_sdn_datasets
from helper_functions.preprocess_SHRED import (
    convert_to_anomaly,
    create_shred_sequences,
    split_ordered_data,
)
from helper_functions.saving import JsonEncoder

np.random.seed(42)
random.seed(42)

## Set up

### Load data

In [ ]:
dataset = "OFAM3"  # "OFAM3" or "NOAA"
area = "NA"  # "NA" or "world"

In [ ]:
if dataset == "NOAA":
    data, lats, longs, dates = noaa_data.load_data()

    if area != "world":
        data, lats, longs = noaa_data.crop_to_area(data, lats, longs, area=area)

    marker_size = 25

elif dataset == "OFAM3":
    data, dates, lats, longs = ofam3_data.load_files(file_path=f"Data/OFAM3/{area}/")
    marker_size = 1

print(data.shape, len(longs), len(lats))

### Set up data preprocessing parameters

In [ ]:
## CHANGE THESE

sensor_placement = "random"  # "QR" or "random"
num_sensors = 100  # 3 for NOAA experiments, 100 and 47 for OFAM3
lags = 52
anomaly = False

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, train_dates, val_dates, test_dates = split_ordered_data(data, dates, all_dates=True)
if anomaly:
    train_anomaly, val_anomaly, test_anomaly, bulk_mean = convert_to_anomaly((train_data, val_data, test_data))

In [ ]:
print("Training shape:\t\t", train_data.shape)
print("Validation shape:\t", val_data.shape)
print("Test shape:\t\t", test_data.shape)
print("Total number of time samples:\t", len(train_data)+len(test_data)+len(val_data))

In [ ]:
# Plots for santity checks:
plot_index = 5  # 5 for NOAA matches 683 for OFAM3 in dates
sst_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": train_data[plot_index]
})
plot_sst_map(sst_df, cmap="plasma", area=area, marker_size=marker_size, background="white")

if anomaly:
    anomaly_df = pd.DataFrame({
        "Longitude": longs,
        "Latitude": lats,
        "SST": train_anomaly[plot_index]
    })
    plot_sst_map(anomaly_df, cmap="viridis", title=f"SST Anomaly, Date: {train_dates[plot_index]}", area=area)

    bulk_df = pd.DataFrame({
        "Longitude": longs,
        "Latitude": lats,
        "SST": bulk_mean
    })
    plot_sst_map(bulk_df, cmap="plasma", title=f"Bulk mean SST, Dates: {train_dates[0]} to {train_dates[-1]}", area=area)

In [ ]:
# Normalise data
if anomaly:
    sc = MinMaxScaler()
    sc = sc.fit(train_anomaly)  # Computes min and max from training data only (prevent data leakage)
    train_transformed = sc.transform(train_anomaly)  
    val_transformed = sc.transform(val_anomaly)
    test_transformed = sc.transform(test_anomaly)
else:
    sc = MinMaxScaler()
    sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
    train_transformed = sc.transform(train_data)
    val_transformed = sc.transform(val_data)
    test_transformed = sc.transform(test_data)

NOTE: be really careful to restart the notebook and check that the sensor locs are the same as using in the metadata when producing graphs with the below code

In [ ]:
# Sensor placements
if sensor_placement == "random":
    # Set sensor locations
    sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)

elif sensor_placement == "QR":
    sensor_locations, _ = qr_place(train_data, num_sensors)

else:
    raise ValueError("Incorrect placement of sensors specified")

sensor_coords=(longs[sensor_locations], lats[sensor_locations])
print(f"Sensor location indexes will be: {sensor_locations}")

## Model training

Set model parameters and version for saving

NOTE: After running the above, run from here if want to use the same dataset set up for multiple models

In [ ]:
# Model parameters
version = "051026-2"
model_type = "SDN"
sequence_model = "LSTM"  # "GRU" or "LSTM", only for SHRED models
hidden_dim = 128

# Storage
if model_type == "SHRED":
    model_name = f"{model_type}-{sequence_model}"
elif model_type == "SDN":
    model_name = model_type
exp_name = f"{dataset}/{area}/{sensor_placement}/{version}/{model_name}"
print("Experiment: ", exp_name)

#### Build sequences

train_dataset, val_dataset, or test_dataset will contain a torch Dataset of input (X) and output (y) values

For OFAM3 data:

SHRED:
- X will have shape (1356, 52, 100) -> each week (of 1356) gives the preceding year (aka lag of 52 weeks) of SST values at 100 sensor locations
- y will have shape (1356, 59063) -> each week has the ground truth of the SST values at all 59063 possible locations

SDN:
- X will have shape (1408, 100) -> for each week (of 1408) use the SST values at 100 sensor locations
- y will have shape (1408, 59063) -> each week has the ground truth of the SST values at all 59063 possible locations

- an extra test dataset (test_dataset_match_shred) is produced which only tests at the 183 end points of each year lag

In [ ]:
# Build sequences
if model_type == "SHRED":
    train_dataset = create_shred_sequences(train_transformed, sensor_locations, lags=lags)
    val_dataset = create_shred_sequences(val_transformed, sensor_locations, lags=lags)
    test_dataset = create_shred_sequences(test_transformed, sensor_locations, lags=lags)

    # Get dates corresponding to the predictions made by the model
    train_lag_dates = train_dates[np.array(range(len(train_dates)-lags)) + lags - 1]
    val_lag_dates = val_dates[np.array(range(len(val_dates)-lags)) + lags - 1]
    test_lag_dates = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

elif model_type == "SDN":
    train_dataset = create_sdn_datasets(train_transformed, sensor_locations, test_lags=lags)
    val_dataset = create_sdn_datasets(val_transformed, sensor_locations, test_lags=lags)
    test_dataset = create_sdn_datasets(test_transformed, sensor_locations)

    # Generate test set to match SHRED test set
    # (use final sensor results for what each sequence would be if using SHRED)
    test_dataset_match_shred = create_sdn_datasets(test_transformed, sensor_locations, test_lags=lags)
    test_lag_dates = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

In [ ]:
print(len(train_dataset))
print(train_dataset.X.shape, train_dataset.Y.shape)
print(test_dataset.X.shape, test_dataset.Y.shape)
# print(test_dataset_match_shred.X.shape, test_dataset_match_shred.Y.shape)

In [ ]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))
print(len(train_dataset) + len(val_dataset) + len(test_dataset))

In [ ]:
if model_type == "SHRED":
    plot_index = 0
    plot_data = test_dataset[plot_index]
    plt.rcParams.update({'font.size': 16})
    if num_sensors <= 5:
        colours = ["#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100"]
    else:
        colours = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)])
                   for i in range(num_sensors)]
    for i in range(num_sensors):
        # plot_data[0] -> test sequences
        # plot_data[0].T -> for each sensor, timeseries of SST values across test dataset
        if num_sensors > 5:
            plt.plot(plot_data[0].T[i], c=colours[i])
        else:
            plt.plot(plot_data[0].T[i], c=colours[i], label=f"Sensor {i+1}")
    plt.ylabel("SST ($\\degree$C)")
    plt.xlabel("Week index")
    plt.grid()
    plt.xlim(0, 52)
    if num_sensors <= 5:
        plt.legend()  #bbox_to_anchor=(1,1)
    plt.show()

    sst_df = pd.DataFrame({
        "Longitude": longs,
        "Latitude": lats,
        "SST": train_data[plot_index]
    })
    plot_sst_map(sst_df, cmap="plasma", area=area, marker_size=marker_size, background="white")

In [ ]:
sensor_df = pd.DataFrame({
    "Longitude": sensor_coords[0],
    "Latitude": sensor_coords[1],
    "SST": test_data[plot_index][sensor_locations]
})
# plot_sensors(sensor_coords=sensor_coords)
plot_sst_map(sensor_df, area="NA", background="lightblue", marker_size=5,
            min_sst=min(test_data[plot_index]), max_sst=max(test_data[plot_index]))
            # annotate=[78, 83, 87, 94, 96, 92, 93, 100])  # < OFAM3 QR placement

### Train + test SHRED model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if model_type == "SHRED":
    model = models.SHRED(num_sensors, len(lats), seq_model=sequence_model, hidden_size=hidden_dim,
                        hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
elif model_type == "SDN":
    model = models.SDN(num_sensors, len(lats), l1=350, l2=400, dropout=0.1).to(device)

validation_errors = models.fit(model, train_dataset, val_dataset, batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
test_recons = sc.inverse_transform(model(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

if model_type == "SDN":
    test_recon_lags = sc.inverse_transform(model(test_dataset_match_shred.X).detach().cpu().numpy())
    ground_truth_lags = sc.inverse_transform(test_dataset_match_shred.Y.detach().cpu().numpy())
    print('Test (matching SHRED lags) Reconstruction Error: ')
    print(np.linalg.norm(test_recon_lags - ground_truth_lags)/ np.linalg.norm(ground_truth_lags))

### Save results

In [ ]:
# Make experiment folder
directory = f"Reconstructions/Static/{exp_name}/"
os.makedirs(directory, exist_ok = True)

In [ ]:
if model_type == "SHRED":
    np.save(f"Reconstructions/Static/{exp_name}/recons_n{num_sensors}_l{lags}", test_recons)
    np.save(f"Reconstructions/Static/{exp_name}/truth_n{num_sensors}_l{lags}", test_ground_truth)

elif model_type == "SDN":
    np.save(f"Reconstructions/Static/{exp_name}/recons_n{num_sensors}", test_recons)
    np.save(f"Reconstructions/Static/{exp_name}/truth_n{num_sensors}", test_ground_truth)
    np.save(f"Reconstructions/Static/{exp_name}/recons_n{num_sensors}_l{lags}", test_recon_lags)
    np.save(f"Reconstructions/Static/{exp_name}/truth_n{num_sensors}_l{lags}", ground_truth_lags)

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords)
    }

if anomaly:
    metadata["bulk_mean"] = bulk_mean

In [ ]:
if model_type == "SHRED":
    metadata["test_lag_dates"] = test_lag_dates
    with open(f"Reconstructions/Static/{exp_name}/metadata_n{num_sensors}_l{lags}.json", 'w', encoding='utf-8') as f:
        json.dump(metadata, f, cls=JsonEncoder)

elif model_type == "SDN":
    with open(f"Reconstructions/Static/{exp_name}/metadata_n{num_sensors}.json", 'w', encoding='utf-8') as f:
        json.dump(metadata, f, cls=JsonEncoder)

    metadata["test_lag_dates"] = test_lag_dates
    with open(f"Reconstructions/Static/{exp_name}/metadata_n{num_sensors}_l{lags}.json", 'w', encoding='utf-8') as f:
        json.dump(metadata, f, cls=JsonEncoder)

In [ ]:
print(f"Model trained and reconstructions produced for {exp_name}")

## References

[[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html) NOAA dataset origin, https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html

[[2]](https://github.com/Jan-Williams/pyshred/blob/main/Data/SST_data.mat) Jan P Williams, pyshred repository, NOAA SST dataset, https://github.com/Jan-Williams/pyshred/blob/main/Data/SST_data.mat

[[3]](https://data.csiro.au/collection/csiro%3A60826v9) CSIRO ME4OH OFAM3 dataset, v9, https://data.csiro.au/collection/csiro%3A60826v9

[[4]](https://github.com/Jan-Williams/pyshred/blob/main/models.py) Jan P Williams, pyshred repository, SHRED and SDN model implementations, https://github.com/Jan-Williams/pyshred/blob/main/models.py

[[5]](https://royalsocietypublishing.org/rspa/article/480/2298/20240054/66770/) Jan P Williams, Olivia Zahn, and J Nathan Kutz. Sensing with shallow recurrent decoder networks. Proceedings of the Royal Society A: Mathematical, Physical and Engineering Sciences, 480(2298), 2024, https://royalsocietypublishing.org/rspa/article/480/2298/20240054/66770/.

[[6]](https://ieeexplore.ieee.org/document/10584544) Megan R Ebers, Jan P Williams, Katherine M Steele, and J Nathan Kutz. Leveraging
arbitrary mobile sensor trajectories with shallow recurrent decoder networks for full-state
reconstruction. IEEE Access, 12:97428–97439, 2024, https://ieeexplore.ieee.org/document/10584544.